In [ ]:
# --- Setup : ajoute le dossier parent au path ---
import sys, os
sys.path.insert(0, os.path.abspath(".."))

%load_ext autoreload
%autoreload 2
%matplotlib inline


# 3. Méthodes ABC : Rejet vs Gibbs

Dans ce notebook, nous appliquons l'Approximate Bayesian Computation (ABC) à notre modèle MA(2).
Puisque nous connaissons la vraie distribution grâce au Gold Standard (RWMH), nous allons pouvoir évaluer la qualité de nos approximations ABC.

**Objectifs :**
1. Tester l'ABC par Rejet standard.
2. Tester l'ABC-Gibbs (inspiré de Clarté et al.).
3. Observer l'impact du seuil de tolérance $\epsilon$ sur le temps de calcul et la précision.

In [ ]:
# --- COMMANDES MAGIQUES ---
%load_ext autoreload
%autoreload 2
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Import des modèles et statistiques
import ma_model
import summary_stats as summary_stats
import distances

# Import de nos samplers
from samplers.rwmh import run as run_rwmh
from samplers.abc_reject import ABCRejectSampler
from samplers.abc_gibbs import ABCGibbsSampler

sns.set_theme(style="whitegrid")
rng = np.random.default_rng(19)

## 1. Données Observées et Gold Standard
Nous générons la même série temporelle que précédemment. Pour avoir une base de comparaison immédiate, nous lançons un RWMH rapide (10 000 itérations).

In [ ]:
# 1. Les données
theta_true = np.array([0.6, 0.2])
T = 500
y_obs = ma_model.simulate(theta_true, T, rng)
s_obs = summary_stats.summary_full(y_obs)
print(f"Statistiques observées (rho_1, rho_2) : {s_obs[0]:.4f}, {s_obs[1]:.4f}")

# 2. Le Gold Standard (RWMH)
print("\nGénération du Gold Standard en tâche de fond...")
res_gold = run_rwmh(y_obs, n_iter=10000, proposal_scale=0.12, burnin=2000, show_progress=False)
print("✅ Gold Standard prêt !")

## 2. ABC par Rejet Standard
L'algorithme ABC le plus simple : on tire $\theta$ dans l'a priori, on simule $y_{sim}$, et on accepte si la distance euclidienne entre les autocorrélations est inférieure à $\epsilon$.

*Note sur les wrappers : Nos samplers attendent des fonctions simples `f(theta)`. Nous "emballons" (wrap) les fonctions de notre modèle pour fixer la taille $T$ et le générateur aléatoire.*

In [ ]:
# Wrappers pour adapter les fonctions du binôme à notre ABCRejectSampler
def prior_sampler_single():
    # Tire un seul échantillon et l'extrait du tableau
    return ma_model.sample_prior(np.random.default_rng(), n=1)[0]

def simulator_wrapper(theta):
    # Simule une série de taille T
    return ma_model.simulate(theta, T, np.random.default_rng())

# Instanciation de l'échantillonneur
sampler_reject = ABCRejectSampler(
    prior_sampler=prior_sampler_single,
    simulator=simulator_wrapper,
    summary_stats=summary_stats.summary_full,
    distance_metric=distances.euclidean
)

# --- PARAMÈTRES ABC REJET ---
n_samples_wanted = 500
epsilon_rej = 0.1   # <-- C'est ici qu'il faut jouer ! (Essaie 0.5, puis 0.1, puis 0.05)

print(f"Lancement ABC-Rejet (Cible: {n_samples_wanted} échantillons, Epsilon: {epsilon_rej})...")
res_rej = sampler_reject.sample(y_obs, n_samples=n_samples_wanted, epsilon=epsilon_rej)

print("\n--- Résultats ABC-Rejet ---")
res_rej.summary()

## 3. ABC-Gibbs
Contrairement au rejet standard qui propose un vecteur complet $(\theta_1, \theta_2)$, l'ABC-Gibbs met à jour $\theta_1$ puis $\theta_2$ conditionnellement. Cela augmente drastiquement le taux d'acceptation, surtout en haute dimension.

Il nous faut définir la loi a priori conditionnelle pour que les propositions restent dans le triangle d'inversibilité du MA(2).

In [ ]:
def sample_conditional_prior_ma2(j, theta_curr):
    """Loi uniforme conditionnelle respectant le triangle d'inversibilité."""
    t1, t2 = theta_curr[0], theta_curr[1]
    if j == 0:
        return np.random.uniform(-1 - t2, 1 + t2)
    else:
        return np.random.uniform(abs(t1) - 1, 1)

sampler_gibbs = ABCGibbsSampler(
    conditional_prior_sampler=sample_conditional_prior_ma2,
    simulator=simulator_wrapper,
    summary_stats=summary_stats.summary_full,
    distance_metric=distances.euclidean
)

# --- PARAMÈTRES ABC GIBBS ---
n_iters_gibbs = 500
epsilon_gibbs = 0.1

print(f"Lancement ABC-Gibbs (Itérations: {n_iters_gibbs}, Epsilon: {epsilon_gibbs})...")
res_gibbs = sampler_gibbs.sample(y_obs, theta_init=np.array([0.0, 0.0]), n_samples=n_iters_gibbs, epsilon=epsilon_gibbs)

# On retire un petit burn-in pour le Gibbs
burnin_gibbs = 100
res_gibbs.samples = res_gibbs.samples[burnin_gibbs:]

print("\n--- Résultats ABC-Gibbs ---")
res_gibbs.summary()

## 4. Comparaison Visuelle des Distributions
Superposons les résultats de nos deux méthodes ABC avec la vraie distribution (Gold Standard).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Liste des résultats à tracer
results = {
    "Gold Standard (RWMH)": (res_gold, 'black', 2, 0.0),
    "ABC-Rejet": (res_rej, '#3498db', 1.5, 0.2),
    "ABC-Gibbs": (res_gibbs, '#e74c3c', 1.5, 0.2)
}

for name, (res, color, lw, alpha) in results.items():
    if len(res.samples) > 0:
        sns.kdeplot(res.samples[:, 0], ax=axes[0], label=name, color=color, linewidth=lw, fill=(alpha>0), alpha=alpha)
        sns.kdeplot(res.samples[:, 1], ax=axes[1], label=name, color=color, linewidth=lw, fill=(alpha>0), alpha=alpha)

axes[0].axvline(theta_true[0], color='black', linestyle='--')
axes[1].axvline(theta_true[1], color='black', linestyle='--')

axes[0].set_title(r"Densité marginale de $\theta_1$")
axes[0].legend()

axes[1].set_title(r"Densité marginale de $\theta_2$")
axes[1].legend()

plt.suptitle(f"Comparaison de l'Inférence (Epsilon = {epsilon_rej})", fontsize=14)
plt.tight_layout()
plt.show()